<a href="https://colab.research.google.com/github/AgustinBiasca/Logistic-Regression/blob/main/K_Nearste_Neighbors_SMARKET.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install ISLP

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.3/349.3 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.4/832.4 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 5.5 MB/s eta 0:00:00
  Created wheel for autograd-gamma: filename=autograd_gamma-0.5.0-py3-none-any.whl size=4030 sha256=eb0aede60a05efcb486beb7285c8546e38468bf0c1b86390b331578df43a09ac
  Stored in directory: /root/.cache/pip/wheels/50/37/21/0a719b9d89c635e89ff24bd93b862882ad675279552013b2fb
Successfully built autograd-gamma


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.pyplot import subplots
import seaborn as sns
import statsmodels.api as sm
from ISLP import load_data
from ISLP.models import (ModelSpec as MS,
                         summarize)

from ISLP import confusion_table
from ISLP.models import contrast
from sklearn.discriminant_analysis import \
    (LinearDiscriminantAnalysis as LDA,
     QuadraticDiscriminantAnalysis as QDA)

from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

In [3]:
Smarket = load_data('Smarket')
Smarket.columns

Index(['Year', 'Lag1', 'Lag2', 'Lag3', 'Lag4', 'Lag5', 'Volume', 'Today',
       'Direction'],
      dtype='object')

In [7]:
model = MS(['Lag1','Lag2']).fit(Smarket)

X = model.transform(Smarket)
y = Smarket.Direction == "Up"

train = (Smarket.Year < 2005)

X_train, X_test = X.loc[train] , X.loc[~train]

y_train, y_test = y.loc[train], y.loc[~train]

D = Smarket.Direction

L_train , L_test = D.loc[train], D.loc[~train]

In [10]:
knn1 = KNeighborsClassifier(n_neighbors=1)
X_train, X_test = [M.drop(columns=['intercept'])
                   for M in (X_train, X_test)]


In [11]:
knn1.fit(X_train, L_train)

KNeighborsClassifier(n_neighbors=1)

In [12]:
knn1_pred = knn1.predict(X_test)

confusion_table(knn1_pred , L_test)

Truth,Down,Up
Predicted,,
Down,43,58
Up,68,83


In [14]:
np.mean(knn1_pred == L_test), np.mean(knn1_pred != L_test)

(np.float64(0.5), np.float64(0.5))

Los resultados usando K = 1 no son muy buenos, ya que solo el 50% de las observaciones se predicen correctamente. Por supuesto, puede ser que K = 1 resulte en un ajuste demasiado flexible a los datos.

El valor de 1 es arbitrario, podemos probar con otros valores. Por ejemplo con 3. Vamos a ver si el modelo mejora.

In [18]:
knn3 = KNeighborsClassifier(n_neighbors=3)
knn3_pred = knn3.fit(X_train, L_train).predict(X_test)
np.mean(knn3_pred == L_test)

np.float64(0.5317460317460317)

Vemos una muy sutil mejora, sin embargo, es insuficiente